In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip uninstall -y datasets

Found existing installation: datasets 4.4.1
Uninstalling datasets-4.4.1:
  Successfully uninstalled datasets-4.4.1


In [3]:
!pip install datasets==2.17

INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.6/536.6 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.10.0
    Uninstalling fsspec-2025.10.0:
      Successfully uninstalled fsspec-2025.10.0
  Attempting uninstall: dill
    Found existing installation: dill 0.4.0
    Uninstalling dill-0.4.0:
      Successfully uninstalled dill-0.4.0
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.18
    Uninstalling multiprocess-0.70.18:
      Successfully uninstalled multiprocess-0.70.18
ERROR: pip's dependency resolver does 

In [4]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn as nn
from peft import LoraConfig, get_peft_model, TaskType
from tqdm import tqdm
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import numpy as np

2025-12-06 08:47:05.921502: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765010826.092207      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765010826.143014      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [5]:
# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [6]:
# Load dataset
dataset_fabsa = load_dataset("jordiclive/fabsa")
train_ds = dataset_fabsa["train"]
test_ds = dataset_fabsa["test"]

Generating train split:   0%|          | 0/7930 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1057 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1587 [00:00<?, ? examples/s]

In [7]:
# Extract unique Aspect Labels (ignoring sentiment)
all_aspects = set()

# We iterate over the raw list to avoid tensor errors
for label_entry in train_ds['labels']:
    for aspect, sentiment in label_entry:
        all_aspects.add(aspect)

aspect_list = sorted(list(all_aspects))
num_labels = len(aspect_list)
label2id = {l: i for i, l in enumerate(aspect_list)}
id2label = {i: l for l, i in label2id.items()}

print(f"Found {num_labels} unique categories: {aspect_list}")

Found 12 unique categories: ['Account management: Account access', 'Company brand: Competitor', 'Company brand: General satisfaction', 'Company brand: Reviews', 'Logistics rides: Speed', 'Online experience: App website', 'Purchase booking experience: Ease of use', 'Staff support: Attitude of staff', 'Staff support: Email', 'Staff support: Phone', 'Value: Discounts promotions', 'Value: Price value for money']


In [8]:
# Data Processing (Multi-Hot Encoding)
def encode_data(example):
    # Create a vector of zeros [0, 0, ... 0]
    vec = [0.0] * num_labels 
    
    # Loop through labels, get aspect, ignore sentiment
    for aspect, sentiment in example["labels"]:
        if aspect in label2id:
            idx = label2id[aspect]
            vec[idx] = 1.0
            
    # Tokenize text
    enc = tokenizer(example["text"], padding="max_length", truncation=True, max_length=128)
    
    # Add labels to the encoding
    enc["labels"] = vec
    return enc

In [9]:
# Initialize Tokenizer
model_name = "albert-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

In [10]:
# Apply processing
train_ds = train_ds.map(encode_data, batched=False)
train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# Apply processing
test_ds = test_ds.map(encode_data, batched=False)
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/7930 [00:00<?, ? examples/s]

Map:   0%|          | 0/1587 [00:00<?, ? examples/s]

In [11]:
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=16, shuffle=False)

In [12]:
# Load model
bert = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=num_labels,
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/47.4M [00:00<?, ?B/s]

Some weights of AlbertForSequenceClassification were not initialized from the model checkpoint at albert-base-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
# Apply LoRA
lora_cfg = LoraConfig(
    r=16,          # Rank (Paper uses full fine-tuning, but r=16 is good for LoRA)
    lora_alpha=32,
    target_modules=["query", "key", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

In [14]:
model = get_peft_model(bert, lora_cfg)
model.to(device)

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): AlbertForSequenceClassification(
      (albert): AlbertModel(
        (embeddings): AlbertEmbeddings(
          (word_embeddings): Embedding(30000, 128, padding_idx=0)
          (position_embeddings): Embedding(512, 128)
          (token_type_embeddings): Embedding(2, 128)
          (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0, inplace=False)
        )
        (encoder): AlbertTransformer(
          (embedding_hidden_mapping_in): Linear(in_features=128, out_features=768, bias=True)
          (albert_layer_groups): ModuleList(
            (0): AlbertLayerGroup(
              (albert_layers): ModuleList(
                (0): AlbertLayer(
                  (full_layer_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
                  (attention): AlbertSdpaAttention(
                    (query): lora.Linear(
                      (base_layer)

In [15]:
# Optimizer
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

In [16]:
loss_fn = nn.BCEWithLogitsLoss()

In [17]:
epochs = 10 # FABSA paper suggests training until convergence (usually 5-10 epochs)

print("\nStarting Training...")
for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].float().to(device)
        
        optimizer.zero_grad()
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")


Starting Training...


Epoch 1: 100%|██████████| 496/496 [01:17<00:00,  6.39it/s]


Epoch 1 Loss: 0.3459


Epoch 2: 100%|██████████| 496/496 [01:17<00:00,  6.43it/s]


Epoch 2 Loss: 0.2833


...

Epoch 8 Loss: 0.1622


Epoch 9: 100%|██████████| 496/496 [01:17<00:00,  6.43it/s]


Epoch 9 Loss: 0.1538


Epoch 10: 100%|██████████| 496/496 [01:17<00:00,  6.43it/s]

Epoch 10 Loss: 0.1471


In [18]:
model.eval()
y_true = []
y_pred = []

print("\nStarting Evaluation...")
with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        
        # Sigmoid activation -> Probability
        probs = torch.sigmoid(logits)
        
        # Threshold at 0.5 (Standard for multi-label)
        preds = (probs > 0.3).int().cpu().numpy()
        labels = batch["labels"].cpu().numpy()
        
        y_true.extend(labels)
        y_pred.extend(preds)

# Calculate Metrics
micro_f1 = f1_score(y_true, y_pred, average="micro")
macro_f1 = f1_score(y_true, y_pred, average="macro")
precision = precision_score(y_true, y_pred, average="micro")
recall = recall_score(y_true, y_pred, average="micro")


y_true_arr = np.array(y_true)
y_pred_arr = np.array(y_pred)

print(f"Micro F1:  {micro_f1:.4f}")
print(f"Macro F1:  {macro_f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")


Starting Evaluation...


100%|██████████| 100/100 [00:06<00:00, 14.83it/s]

Micro F1:  0.7657
Macro F1:  0.6865
Precision: 0.7167
Recall:    0.8219


In [19]:
y_true_arr = np.array(y_true)
y_pred_arr = np.array(y_pred)

sample_accuracies = []
for t, p in zip(y_true_arr, y_pred_arr):
    correct = (t * p).sum()               # count correctly predicted labels
    total = t.sum()                       # total actual labels for that sample
    if total == 0:                         # if no true labels exist
        sample_accuracies.append(1.0)      
    else:
        sample_accuracies.append(correct / total)

overall_label_accuracy = np.mean(sample_accuracies)
print(f"Label-wise Sample Accuracy: {overall_label_accuracy:.4f}")


Label-wise Sample Accuracy: 0.8521
